In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc

ROOT = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC"
FIG1_DIR = os.path.join(ROOT, "01_Figure1_python")
FIG_DIR = os.path.join(FIG1_DIR, "Figure1_marker_based_subtypes")

os.makedirs(FIG_DIR, exist_ok=True)

H5AD_FILE = os.path.join(FIG1_DIR, "Step10_adata_all_final_celltype.h5ad")

OUT_H5AD = os.path.join(
    FIG1_DIR,
    "Step10_adata_marker_based_subtypes.h5ad"
)

print(H5AD_FILE)
print(FIG_DIR)

In [ ]:
adata = sc.read_h5ad(H5AD_FILE)

print(adata)
print(adata.obs.columns.tolist())
print(adata.obs["celltype"].value_counts())

In [ ]:
def present_genes(genes):
    return [g for g in genes if g in adata.var_names]

def score_gene_set(score_name, genes):
    genes_use = present_genes(genes)
    print(score_name, genes_use)
    
    if len(genes_use) == 0:
        adata.obs[score_name] = 0.0
    else:
        sc.tl.score_genes(
            adata,
            gene_list=genes_use,
            score_name=score_name,
            use_raw=False
        )

def choose_max_score(df, score_cols, labels, min_delta=0.02):
    arr = df[score_cols].values
    max_idx = arr.argmax(axis=1)
    max_val = arr.max(axis=1)
    second_val = np.partition(arr, -2, axis=1)[:, -2]
    
    labels_out = np.array([labels[i] for i in max_idx], dtype=object)
    labels_out[(max_val - second_val) < min_delta] = "Ambiguous"
    
    return labels_out

In [ ]:
# T cells
SIG_CD4_T = ["CD4", "IL7R", "LTB", "CCR7", "TCF7", "SELL"]
SIG_CD8_T = ["CD8A", "CD8B", "NKG7", "GZMB", "PRF1", "GNLY", "CST7"]
SIG_TREG  = ["FOXP3", "IL2RA", "CTLA4", "TIGIT", "TNFRSF18"]

# TAM
SIG_M1 = ["IL1B", "CXCL9", "CXCL10", "TNF", "FCGR3A", "LST1", "C1QA", "C1QB", "C1QC"]
SIG_M2 = ["CD163", "MRC1", "MSR1", "APOE", "MARCO", "VSIG4", "IL10"]

# Malignant states
SIG_CYCLING = ["MKI67", "TOP2A", "UBE2C", "STMN1", "TYMS", "PCNA"]
SIG_HYPOXIA = ["HIF1A", "VEGFA", "CA9", "LDHA", "SLC2A1", "ENO1", "PGK1", "BNIP3", "NDRG1"]
SIG_EMT     = ["VIM", "FN1", "SNAI2", "ZEB1", "ZEB2", "COL1A1", "COL1A2", "TAGLN", "SPARC"]
SIG_HPC_MAL = ["KRT19", "EPCAM", "SOX9", "PROM1", "ALB", "KRT8", "KRT18"]

score_gene_set("score_CD4_T", SIG_CD4_T)
score_gene_set("score_CD8_T", SIG_CD8_T)
score_gene_set("score_Treg", SIG_TREG)

score_gene_set("score_M1", SIG_M1)
score_gene_set("score_M2", SIG_M2)

score_gene_set("score_Cycling", SIG_CYCLING)
score_gene_set("score_Hypoxia", SIG_HYPOXIA)
score_gene_set("score_EMT", SIG_EMT)
score_gene_set("score_HPC_malignant", SIG_HPC_MAL)

In [ ]:
adata.obs["celltype_marker_subtype"] = adata.obs["celltype"].astype(str)

# T cell 二级注释
t_mask = adata.obs["celltype"].astype(str).eq("T cell")

adata.obs.loc[t_mask, "celltype_marker_subtype"] = choose_max_score(
    adata.obs.loc[t_mask],
    ["score_CD4_T", "score_CD8_T", "score_Treg"],
    ["CD4 T", "CD8 T", "Treg"],
    min_delta=0.03
)

# TAM 二级注释
tam_mask = adata.obs["celltype"].astype(str).eq("TAM")

adata.obs.loc[tam_mask, "celltype_marker_subtype"] = np.where(
    adata.obs.loc[tam_mask, "score_M1"] >= adata.obs.loc[tam_mask, "score_M2"],
    "M1 macrophage",
    "M2 macrophage"
)

# Malignant 不再细分，只保留 Malignant cell
mal_mask = adata.obs["celltype"].astype(str).eq("Malignant cell")
adata.obs.loc[mal_mask, "celltype_marker_subtype"] = "Malignant cell"

# 去掉 Ambiguous，避免图上太乱
adata.obs.loc[
    adata.obs["celltype_marker_subtype"].eq("Ambiguous"),
    "celltype_marker_subtype"
] = "T cell"

adata.obs["celltype_marker_subtype"] = (
    adata.obs["celltype_marker_subtype"]
    .astype(str)
    .fillna("Unknown")
    .astype("category")
)

# 推荐排序
subtype_order = [
    "B cell",
    "CAF",
    "TEC",
    "CD4 T",
    "CD8 T",
    "Treg",
    "M1 macrophage",
    "M2 macrophage",
    "HPC-like",
    "Malignant cell",
    "unclassified"
]

exist_order = [x for x in subtype_order if x in adata.obs["celltype_marker_subtype"].unique()]
adata.obs["celltype_marker_subtype"] = adata.obs["celltype_marker_subtype"].cat.set_categories(
    exist_order,
    ordered=True
)

print(adata.obs["celltype_marker_subtype"].value_counts())

In [ ]:
adata.obs.to_csv(
    os.path.join(FIG_DIR, "Figure1_marker_based_subtype_metadata.csv")
)

adata.write_h5ad(OUT_H5AD)

print("Saved:")
print(OUT_H5AD)
print(FIG_DIR)

In [ ]:
sc.settings.figdir = FIG_DIR
sc.settings.set_figure_params(
    dpi=120,
    dpi_save=300,
    facecolor="white",
    fontsize=10,
    frameon=False
)

In [ ]:
sc.pl.umap(
    adata,
    color="celltype_marker_subtype",
    size=6,
    frameon=False,
    legend_loc="right margin",
    save="_Fig1_marker_based_subtype_umap.pdf"
)

In [ ]:
RAW_FILE = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python/Step9A_adata_all_final_annotation_raw_merged.h5ad"

adata_raw = sc.read_h5ad(RAW_FILE)

print(adata_raw)
print("genes:", adata_raw.n_vars)

In [ ]:
import os
import scanpy as sc

FIG1_DIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"

for f in os.listdir(FIG1_DIR):
    if f.endswith(".h5ad"):
        path = os.path.join(FIG1_DIR, f)
        try:
            ad = sc.read_h5ad(path, backed="r")
            print(f, ad.n_obs, ad.n_vars)
            ad.file.close()
        except Exception as e:
            print(f, "ERROR", e)

In [ ]:
# UMAP 和注释用 Step10
adata_umap = sc.read_h5ad(
    "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python/Step10_adata_all_final_celltype.h5ad"
)

# marker dotplot 表达用 raw full genes
adata_raw = sc.read_h5ad(
    "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python/Step5A_adata_all_raw_merged.h5ad"
)

print(adata_umap)
print(adata_raw)

In [ ]:
adata_raw.obs["celltype"] = adata_umap.obs.loc[
    adata_raw.obs_names, "celltype"
].values

if "celltype_marker_subtype" in adata_umap.obs.columns:
    adata_raw.obs["celltype_marker_subtype"] = adata_umap.obs.loc[
        adata_raw.obs_names, "celltype_marker_subtype"
    ].values
else:
    adata_raw.obs["celltype_marker_subtype"] = adata_raw.obs["celltype"].astype(str)

In [ ]:
adata_raw.obs["celltype_marker_subtype"] = adata_raw.obs["celltype"].astype(str)

adata_raw.obs.loc[
    adata_raw.obs["celltype"].astype(str).eq("Malignant cell"),
    "celltype_marker_subtype"
] = "Malignant cell"

adata_raw.obs["celltype_marker_subtype"] = adata_raw.obs["celltype_marker_subtype"].astype("category")

In [ ]:
marker_genes = [

    # B cell
    "MS4A1", "CD79A", "CD79B",

    # CAF
    "COL1A1", "COL1A2", "DCN", "LUM",

    # TEC
    "PECAM1", "VWF", "KDR", "EMCN",

    # CD4 T
    "CD3D", "CD3E", "TRAC",
    "IL7R", "CCR7", "LTB",

    # CD8 T
    "CD8A", "CD8B", "NKG7", "GZMK", "GZMB",

    # Treg
    "FOXP3", "IL2RA", "CTLA4", "TIGIT",

    # TAM
    "C1QA", "C1QB", "C1QC",
    "LST1", "TYROBP", "FCER1G",

    # HPC-like
    "KRT19", "SOX9",

    # Malignant
    "EPCAM", "KRT8", "KRT18", "KRT19"
]

marker_genes = [g for g in marker_genes if g in adata_raw.var_names]

sc.pl.dotplot(
    adata_raw,
    var_names=marker_genes,
    groupby="celltype_marker_subtype",
    standard_scale="var",
    dendrogram=False,
    swap_axes=False,
    save="_Fig1_marker_dotplot_more_specific.pdf"
)


In [ ]:
import os
import scanpy as sc
import matplotlib.pyplot as plt

# ============================================================
# Figure save directory
# ============================================================

FIG_DIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"
os.makedirs(FIG_DIR, exist_ok=True)

# ============================================================
# More specific marker genes for Figure1
# ============================================================

marker_genes = [

    # =========================
    # B cell
    # =========================
    "MS4A1", "CD79A", "CD79B",

    # =========================
    # CAF
    # =========================
    "COL1A1", "COL1A2", "DCN",

    # =========================
    # TEC
    # =========================
    "PECAM1", "VWF", "KDR", "EMCN",

    # =========================
    # CD4 T
    # =========================
    "CD3D", "CD3E", "TRAC",
    "IL7R", "CCR7", "LTB",

    # =========================
    # CD8 T
    # =========================
    "CD8A", "CD8B",
    "NKG7", "GZMK", "GZMB",

    # =========================
    # Treg
    # =========================
    "FOXP3", "IL2RA",
    "CTLA4", "TIGIT",

    # =========================
    # TAM
    # =========================
    "C1QA", "C1QB", "C1QC",
    "LST1", "TYROBP", "FCER1G",

    # =========================
    # HPC-like
    # =========================
    "KRT19", "SOX9",

    # =========================
    # Malignant epithelial
    # =========================
    "EPCAM", "KRT8", "KRT18"
]

# ============================================================
# Keep genes existing in raw matrix
# ============================================================

marker_genes = [
    g for g in marker_genes
    if g in adata_raw.var_names
]

print("Final marker genes:")
print(marker_genes)

# ============================================================
# Plot dotplot
# ============================================================

dp = sc.pl.dotplot(
    adata_raw,
    var_names=marker_genes,
    groupby="celltype_marker_subtype",
    standard_scale="var",
    dendrogram=False,
    swap_axes=False,
    figsize=(18, 5),
    cmap="Reds",
    show=False
)

# ============================================================
# Save figure
# ============================================================

out_pdf = os.path.join(
    FIG_DIR,
    "Fig1_marker_dotplot_more_specific.pdf"
)

plt.savefig(
    out_pdf,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Figure saved to:\n{out_pdf}")